In [ ]:
%pip install -q kaggle

In [ ]:
import torch
from torch import nn
from torchvision.datasets import Imagenette

net = nn.Sequential(
    nn.Conv2d(3, 96, kernel_size=11, stride=4, padding=1), nn.ReLU(),
    nn.MaxPool2d(kernel_size=3, stride=2),
    # 减小卷积窗口，使用填充为2来使得输入与输出的高和宽一致，且增大输出通道数
    nn.Conv2d(96, 256, kernel_size=5, padding=2), nn.ReLU(),
    nn.MaxPool2d(kernel_size=3, stride=2),
    # 使用三个连续的卷积层和较小的卷积窗口。
    # 除了最后的卷积层，输出通道的数量进一步增加。
    # 在前两个卷积层之后，汇聚层不用于减少输入的高度和宽度
    nn.Conv2d(256, 384, kernel_size=3, padding=1), nn.ReLU(),
    nn.Conv2d(384, 384, kernel_size=3, padding=1), nn.ReLU(),
    nn.Conv2d(384, 256, kernel_size=3, padding=1), nn.ReLU(),
    nn.MaxPool2d(kernel_size=3, stride=2),
    nn.Flatten(),
    nn.Linear(6400, 4096), nn.ReLU(),
    nn.Dropout(p=0.5),
    nn.Linear(4096, 4096), nn.ReLU(),
    nn.Dropout(p=0.5),
    # 最后是输出层。由于这里使用imagenette，所以用类别数为10，而非论文中的1000
    nn.Linear(4096, 10))

from torchvision import datasets
from torchvision import transforms
from torch.utils.data import DataLoader

data_root = './Imagenet-100'

######
######
#为什么要标准化，这是进行的什么标准化
#imagenet中的统计值
image_mean = [0.485,0.456,0.406]
image_std = [0.229,0.224,0.225]

# 训练集的数据处理
##复现文章中随即水平翻转和随机裁剪
train_transform = transforms.Compose([
    # 随机裁剪出224×224图像，同时包含随机缩放
    transforms.RandomResizedCrop(224),

    # 以0.5的概率进行水平翻转
    transforms.RandomHorizontalFlip(p=0.5),

    # 将PIL图像转换为Tensor
    transforms.ToTensor(),

    # 对三个通道分别标准化
    transforms.Normalize(
        mean=image_mean,
        std=image_std
    )
])

# 验证集的数据处理
####
####
#不知道为什么训练集要这样弄，原文是如何处理的
val_transform = transforms.Compose([
    # 先保持比例，将短边缩放到256
    transforms.Resize(256),

    # 从图像中心裁剪224×224
    transforms.CenterCrop(224),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=image_mean,
        std=image_std
    )
])

#创建训练集和交叉验证集
train_dataset = Imagenette(root='./data', 
        split='train', 
        download=True,
        size = 'full',
        transform = train_transform)
val_dataset = Imagenette(root='./data', 
        split='val', 
        download=True,
        size = 'full',
        transform = val_transform)
#加载训练数据集和测试数据集
batch_size = 32
#保持打乱，
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    drop_last=False
)
val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    drop_last=False
)


In [ ]:
from pathlib import  Path
from torch.utils.tensorboard import SummaryWriter

net = net.to(device)
#损失函数为交叉熵函数
loss_fn = nn.CrossEntropyLoss()
#向量参数为0.9，正则化参数为0.0005
optimizer = torch.optim.SGD(
    params=net.parameters(),
    lr=0.01,
    momentum=0.9,
    weight_decay=5e-3
)
#模仿原文中，不收敛后调低学习率，每10个epoch*0。1
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer=optimizer,
    step_size=10,
    gamma=0.1
)
writer = SummaryWriter(
    log_dir="./logs/alexnet_imagenette"
)
save_dir = Path("./alexnet/checkpoints")
save_dir.mkdir(parents=True, exist_ok=True)
use_amp = device.type == "cuda"
######
####
#混合精度训练？？
scaler = torch.amp.GradScaler(
    device="cuda",
    enabled=use_amp
)


epochs = 30
total_train_step = 0
best_val_accuracy = 0.0


In [ ]:
for epoch in range(epochs):
    print(f'\n=======epoch{epoch+1}/{epochs}============')
    net.train()
    train_loss_sum = 0.0
    train_correct = 0
    train_sample_count = 0
    for batch_index, (images, targets) in enumerate(train_loader):
        # 将图像和标签移动到GPU
        images = images.to(
            device,
            non_blocking=True
        )

        targets = targets.to(
            device,
            non_blocking=True
        )

        # 清除上一轮保存的梯度
        optimizer.zero_grad(set_to_none=True)

        # 混合精度前向传播
        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=use_amp
        ):
            outputs = net(images)
            loss = loss_fn(outputs, targets)

        # 混合精度反向传播
        scaler.scale(loss).backward()

        # 更新模型参数
        scaler.step(optimizer)

        # 更新缩放系数
        scaler.update()

        # ----------------------------------------------------
        # 统计损失
        # ----------------------------------------------------
        batch_size_now = images.size(0)

        train_loss_sum += loss.item() * batch_size_now
        train_sample_count += batch_size_now

        # outputs的形状为：
        # [batch_size, 10]
        #
        # dim=1表示在10个类别得分中寻找最大值
        predictions = outputs.argmax(dim=1)

        train_correct += (
            predictions == targets
        ).sum().item()

        total_train_step += 1

        # 每50个batch打印并记录一次
        if total_train_step % 50 == 0:
            current_train_accuracy = (
                train_correct / train_sample_count
            )

            print(
                f"训练步数：{total_train_step}，"
                f"当前Loss：{loss.item():.4f}，"
                f"累计准确率：{current_train_accuracy:.4f}"
            )

            writer.add_scalar(
                "Batch/train_loss",
                loss.item(),
                total_train_step
            )

            writer.add_scalar(
                "Batch/train_accuracy",
                current_train_accuracy,
                total_train_step
            )


    # 计算整个epoch的平均训练损失
    epoch_train_loss = (
        train_loss_sum / train_sample_count
    )

    epoch_train_accuracy = (
        train_correct / train_sample_count
    )
    # ========================================================
    # 二、验证阶段
    # ========================================================
    net.eval()

    val_loss_sum = 0.0
    val_correct = 0
    val_sample_count = 0

    # 验证时不计算梯度
    with torch.no_grad():
        for images, targets in val_loader:
            images = images.to(
                device,
                non_blocking=True
            )

            targets = targets.to(
                device,
                non_blocking=True
            )

            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=use_amp
            ):
                outputs = net(images)
                loss = loss_fn(outputs, targets)

            batch_size_now = images.size(0)

            val_loss_sum += loss.item() * batch_size_now
            val_sample_count += batch_size_now

            predictions = outputs.argmax(dim=1)

            val_correct += (
                predictions == targets
            ).sum().item()


    epoch_val_loss = (
        val_loss_sum / val_sample_count
    )

    epoch_val_accuracy = (
        val_correct / val_sample_count
    )
   #更新学习率
    scheduler.step()
    # ========================================================
    # 记录到TensorBoard
    # ========================================================
    writer.add_scalars(
        "Epoch/loss",
        {
            "train": epoch_train_loss,
            "val": epoch_val_loss
        },
        epoch + 1
    )

    writer.add_scalars(
        "Epoch/accuracy",
        {
            "train": epoch_train_accuracy,
            "val": epoch_val_accuracy
        },
        epoch + 1
    )
    # ========================================================
    # 六、保存验证集准确率最高的模型
    # ========================================================
    if epoch_val_accuracy > best_val_accuracy:
        best_val_accuracy = epoch_val_accuracy

        best_model_path = save_dir / "alexnet_best.pth"

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": net.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "val_accuracy": epoch_val_accuracy
            },
            best_model_path
        )

        print(
            f"保存新的最优模型，"
            f"验证准确率：{best_val_accuracy:.4f}"
        )


# 关闭TensorBoard写入器
writer.close()

#归一化，1*1卷积